In [1]:
# ==========================================
# NB_BRONZE: RAW -> BRONZE (HISTORICAL SNAPSHOT EDITION)
# ==========================================
from pyspark.sql.functions import *
from datetime import datetime

print("==================================================")
print("BẮT ĐẦU INGESTION TỪ RAW SANG BRONZE")
print("==================================================")

# --- BƯỚC 1: LẤY THỜI GIAN THỰC TẠI THỜI ĐIỂM CHẠY PIPELINE ---
# Tự động sinh chuỗi định dạng: YYYYMMDD_HHMMSS (Ví dụ: 20260616_211530)
current_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"[TIME] Phiên bản thời gian của Batch dữ liệu này: {current_timestamp}")


# --- BƯỚC 2: ĐỌC DỮ LIỆU TỪ NGUỒN RAW CSV ---
# Cấu hình đầy đủ các option để tránh lỗi lệch dòng đối với dữ liệu text có chứa dấu xuống dòng
sales_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("multiLine", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("inferSchema", "true") \
    .load("Files/Raw/sales/sales.csv")

exchange_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("Files/Raw/exchange_rate/exchange_rate.csv")

# Thống kê nhanh số lượng dòng vừa đọc được từ nguồn thô
total_sales_raw = sales_df.count()
total_exchange_raw = exchange_df.count()

print(f"[LOG] Đọc thành công {total_sales_raw} dòng từ mindx_raw_sales_data.csv")
print(f"[LOG] Đọc thành công {total_exchange_raw} dòng từ exchange_rate_2425.csv")


# --- BƯỚC 3: GHI XUỐNG TẦNG BRONZE LAYER (MỖI NGÀY 1 THƯ MỤC RIÊNG - KHÔNG GHI ĐÈ) ---
# Thêm biến thời gian vào đường dẫn lưu trữ để phân tách các phiên bản dữ liệu thô
sales_bronze_path = f"Files/Bronze/sales/snapshot_{current_timestamp}"
exchange_bronze_path = f"Files/Bronze/exchange_rate/snapshot_{current_timestamp}"

sales_df.write.mode("overwrite").parquet(sales_bronze_path)
exchange_df.write.mode("overwrite").parquet(exchange_bronze_path)

print(f"[LOG] Đã ghi dữ liệu Parquet thành công vào: {sales_bronze_path}")
print(f"[LOG] Đã ghi dữ liệu Parquet thành công vào: {exchange_bronze_path}")


# --- BƯỚC 4: GHI NHẬT KÝ VẬN HÀNH (METADATA AUDIT LOGGING) ---
notebook_name = "NB_BRONZE"
status = "SUCCESS"
execution_time = datetime.now()

log_data = [(notebook_name, status, execution_time, total_sales_raw, 0)]
log_schema = ["notebook_name", "status", "execution_at", "processed_records", "quarantine_records"]

pipeline_log_df = spark.createDataFrame(log_data, schema=log_schema)
pipeline_log_df.write.format("delta").mode("append").saveAsTable("sys_pipeline_audit_logs")

print("==================================================")
print("HOÀN THÀNH TẦNG BRONZE LAYER (LƯU TRỮ LỊCH SỬ THÀNH CÔNG)")
print("==================================================")

StatementMeta(, f2a506db-8900-4cc4-b9af-7add2ca80734, 3, Finished, Available, Finished, False)

BẮT ĐẦU INGESTION TỪ RAW SANG BRONZE
[TIME] Phiên bản thời gian của Batch dữ liệu này: 20260620_120433
[LOG] Đọc thành công 5250 dòng từ mindx_raw_sales_data.csv
[LOG] Đọc thành công 24 dòng từ exchange_rate_2425.csv
[LOG] Đã ghi dữ liệu Parquet thành công vào: Files/Bronze/sales/snapshot_20260620_120433
[LOG] Đã ghi dữ liệu Parquet thành công vào: Files/Bronze/exchange_rate/snapshot_20260620_120433
HOÀN THÀNH TẦNG BRONZE LAYER (LƯU TRỮ LỊCH SỬ THÀNH CÔNG)
